# 🧠 Gemma 4 Trace-Tracking Synthesis Agent (Budget-Fit 0.18 Architecture)
### High-Resolution Autonomous Bug Fixing on SWE-bench | Aligned with Top Leaderboard 0.18 / 0.15

This notebook packages **`submission.zip`** using the **Trace-Tracking Hypothesis Synthesis Architecture** combined with the measured **Budget-Fit parameters** that achieved **0.18 and 0.15** on the competition leaderboard.

### 🎯 Key Empirical Breakthroughs Implemented:
1. **Trace Tracking & Hypothesis Formulation**: The agent traces backwards from error symptoms to root causes and forms 1–2 concrete hypotheses before editing.
2. **The 240-Second Harness Verification Timeout (`timeout_seconds: 240`)**: Fixes the critical bug where Phase 2 verification pytest runs timed out on test suites like `requests` (which need ~100s).
3. **The 12-Call Invariant (`edit by call 12`)**: Hard rule stopping runaway exploratory reading. Runs without edits by call 12 fail 87% of the time.
4. **Budget-Fit Ceiling (`8 minutes / 28 tool calls`)**: Allows deep reasoning on solvable tasks while preventing turn burnout. Total 120-task run finishes in ~7–8 hours (safely under the 12h/9h limit).
5. **Targeted-Testing Only & Anti-Pattern Shields**: Never runs bare pytest, never fixes broken existing repo tests, protects test files from modification.


In [ ]:
import os
import zipfile
import hashlib
from pathlib import Path

# Output destination
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
ZIP_PATH = OUT_DIR / 'submission.zip'

# The 4 canonical synthesized submission files
FILES = {
  "agent.yaml": 'name: swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n',
  "configs/sampling.yaml": 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n',
  "eval_config.yaml": 'evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 28\n  max_time_minutes: 8\n  max_turns: 80\n',
  "prompts/system.md": 'You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.\n\n## Core Objective: Fast, Minimal, and Precise Fixes\nAim to understand, resolve, and submit the fix in the minimum number of tool calls (under 8–10 turns). Move directly from the problem statement to the relevant files, apply the solution, verify with a targeted test, and submit.\n\n## Workflow: 5-Stage Trace Tracking & Hypothesis Synthesis\n\n### 1. Trace Tracking & Error Hypothesis Formulation\n- Extract filenames, functions, classes, CLI subcommands, or error messages directly from the problem statement.\n- Trace the failure path backwards: Symptom / Error Message -> Call Site -> Faulty Function / Missing Validation.\n- Formulate 1–2 concrete failure hypotheses before editing:\n  - Hypothesis A: Missing validation or guard condition (e.g. rejecting CRLF, null bytes, or invalid bounds).\n  - Hypothesis B: Missing parameter / argument in function signature or schema serialization.\n  - Hypothesis C: Type mismatch, missing branch, or edge-case handling.\n- Read only the specific target files and lines using `read_file` with tight line ranges. Do not wander across unrelated files.\n- If the problem statement does not provide explicit file paths, search for its most specific identifier or error string with `git grep -n "text" -- \'*.py\' | head -20`.\n- Do not use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`: they only accept exact fully-qualified ids and usually return nothing.\n\n### 2. Hypothesis Reproduction\n- When feasible, confirm your failure hypothesis with a tiny reproducer: write a 3-line script to `/tmp/repro.py` showing the bug and run `python /tmp/repro.py`.\n- Confirm the failure before editing. Never leave scratch files in `/workspace`.\n\n### 3. Implement the Solution Directly (The 12-Call Invariant)\n- By your 12th tool call you must have edited a source file. Runs with no source edit by call 12 fail 87% of the time.\n- Apply the minimal necessary fix directly to the source files using `edit_file`. Copy `old_string` verbatim including leading indentation. Keep changes short and unique.\n- Strictly adhere to specified error strings, exception types, HTTP status codes, and exact API signatures.\n- For documentation code tasks (e.g. FastAPI), edit executable code under `docs_src/`.\n\n### 4. Run Targeted Tests Only (Existing Tests May Be Broken)\n- **Compile Check**: Run `python3 -m py_compile <file>` immediately after editing.\n- **Rerun Reproducer**: Run `python /tmp/repro.py` to confirm the bug is fixed.\n- **Run ONLY Targeted Tests**: Run only the specific test file or test method directly verifying the bug or feature you modified (e.g. `pytest tests/test_target.py -k test_feature` or `python3 -m unittest tests.test_target`).\n- **Be Aware That Existing Tests May Be Broken**: Many repositories contain pre-existing test breakages, missing test data fixtures (e.g. `/test_data`), or environment import errors unrelated to your task.\n- **Do NOT Attempt to Fix Existing Tests**: If an existing test fails due to pre-existing repository issues or missing fixtures, IGNORE IT. Never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.\n- **STRICT RULE: NEVER Run Bare Pytest or Full-Repo Sweeps**: NEVER run bare `pytest`, `pytest .`, `python3 -m unittest discover`, or full-repo test suites without specifying a target file. Full test suites take several minutes, cause catastrophic timeouts, and exhaust your turn and time budgets.\n- If you need to locate the test file, find it explicitly with `find tests -name "*<name>*.py"` instead of running the test runner across the repo.\n\n### 5. Immediate Patch Submission\n- Once your targeted test passes:\n  1. Call `submit_patch` immediately.\n  2. Verify `patch_size > 0` and `files_changed > 0`.\n  3. Output a short summary of the fix to end the session.\n\n## Budget Discipline: 28 tool calls, 8 minutes\n- You have at most 28 tool calls and 8 minutes for this task. When either runs out, the working tree is graded as it is, so an edit made in time counts even if you never reach `submit_patch`.\n- Use at most about 10 tool calls to find and read the code.\n- By your 12th tool call you must have edited a source file. If you have not, stop exploring and make your best-guess edit in the most likely place now. Runs that are still only reading after 12 calls almost never succeed; more reading does not help.\n- After your edit: one compile check or one targeted test, a correction if needed, then `submit_patch`. Do not start new exploration after your 20th tool call.\n\n## Anti-Patterns to Avoid\n- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.\n- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) — always specify the exact test file path.\n- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** — your task is strictly to implement the fix for the reported issue in source code.\n- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed.\n- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.\n- Do NOT refactor or reformat unrelated functions or files.\n- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications.\n',
}

# Package directly into submission.zip
ZIP_PATH.unlink(missing_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for rel_path, content in sorted(FILES.items()):
        zf.writestr(rel_path, content)

# Validate archive integrity
with zipfile.ZipFile(ZIP_PATH) as zf:
    assert zf.testzip() is None, 'Corrupt zip archive!'
    namelist = zf.namelist()
    assert 'agent.yaml' in namelist, 'Missing agent.yaml!'
    print(f'[+] Successfully packaged submission.zip ({ZIP_PATH.stat().st_size:,} bytes):')
    for name in namelist:
        info = zf.getinfo(name)
        print(f'     - {name} ({info.file_size:,} bytes)')

sha256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print(f'[+] SHA-256 Digest: {sha256}')
print(f'[+] Target Leaderboard Calibration: 0.15 – 0.18+ (Budget-Fit + Trace-Tracking Synthesis)')
